# 🇨🇲 Projet 02 — Mobile Money Analytics Cameroon

## Workflow

**SQLite → SQL → Python/Pandas → Visualisation → Power BI → Recommandations métier**

Ce projet utilise **10 000 transactions synthétiques**.  
Aucune donnée réelle de MTN, Orange, d'un client ou d'un agent n'est utilisée.

## Problématique

> Comment une plateforme Mobile Money peut-elle améliorer son activité, son revenu net et sa qualité de service grâce aux données ?


# 0. Connexion à SQLite

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Chemin portable : fonctionne depuis notebooks/ ou depuis la racine du dépôt.
_candidates = [
    Path("../data/database/mobile_money_cameroon.db"),
    Path("data/database/mobile_money_cameroon.db"),
]
DB_PATH = next((p.resolve() for p in _candidates if p.exists()), None)
if DB_PATH is None:
    raise FileNotFoundError(
        "Base SQLite introuvable. Lancez le notebook depuis la racine du dépôt ou depuis notebooks/."
    )

con = sqlite3.connect(DB_PATH)
print("Connexion réussie :", DB_PATH)


# PARTIE 1 — SQL

Ici, **SQL réalise l'extraction et les agrégations**. Python sert à envoyer les requêtes et afficher les résultats.


## 1.1 Aperçu — SELECT

In [ ]:
query = """
SELECT *
FROM vw_mobile_money_analysis
LIMIT 10;
"""
pd.read_sql_query(query, con)

## 1.2 KPI globaux — COUNT / SUM / AVG

In [ ]:
query = """
SELECT
    COUNT(*) AS transactions,
    SUM(CASE WHEN status='Success' THEN 1 ELSE 0 END) AS transactions_reussies,
    ROUND(100.0 * SUM(CASE WHEN status='Success' THEN 1 ELSE 0 END) / COUNT(*), 2) AS taux_succes_pct,
    ROUND(SUM(CASE WHEN status='Success' THEN amount_fcfa ELSE 0 END),0) AS valeur_fcfa,
    ROUND(AVG(CASE WHEN status='Success' THEN amount_fcfa END),0) AS montant_moyen_fcfa,
    SUM(fee_fcfa) AS frais_fcfa,
    SUM(agent_commission_fcfa) AS commissions_agents_fcfa,
    SUM(net_revenue_fcfa) AS revenu_net_fcfa
FROM fact_transactions;
"""
kpi_sql = pd.read_sql_query(query, con)
kpi_sql

## 1.3 Performance par opérateur — GROUP BY

In [ ]:
query = """
SELECT
    operator,
    COUNT(*) AS transactions,
    ROUND(SUM(CASE WHEN status='Success' THEN amount_fcfa ELSE 0 END),0) AS valeur_fcfa,
    SUM(net_revenue_fcfa) AS revenu_net_fcfa,
    ROUND(100.0 * SUM(CASE WHEN status='Success' THEN 1 ELSE 0 END) / COUNT(*),2) AS taux_succes_pct
FROM fact_transactions
GROUP BY operator
ORDER BY valeur_fcfa DESC;
"""
operator_sql = pd.read_sql_query(query, con)
operator_sql

## 1.4 Types de transactions

In [ ]:
query = """
SELECT
    transaction_type,
    COUNT(*) AS transactions,
    ROUND(SUM(CASE WHEN status='Success' THEN amount_fcfa ELSE 0 END),0) AS valeur_fcfa,
    ROUND(AVG(CASE WHEN status='Success' THEN amount_fcfa END),0) AS montant_moyen_fcfa,
    SUM(net_revenue_fcfa) AS revenu_net_fcfa
FROM fact_transactions
GROUP BY transaction_type
ORDER BY valeur_fcfa DESC;
"""
type_sql = pd.read_sql_query(query, con)
type_sql

## 1.5 Villes — GROUP BY

In [ ]:
query = """
SELECT
    city,
    COUNT(*) AS transactions,
    ROUND(SUM(CASE WHEN status='Success' THEN amount_fcfa ELSE 0 END),0) AS valeur_fcfa,
    SUM(net_revenue_fcfa) AS revenu_net_fcfa
FROM fact_transactions
GROUP BY city
ORDER BY valeur_fcfa DESC;
"""
city_sql = pd.read_sql_query(query, con)
city_sql

## 1.6 Segments clients — JOIN

In [ ]:
query = """
SELECT
    c.client_segment,
    COUNT(*) AS transactions,
    COUNT(DISTINCT f.client_id) AS clients_actifs,
    ROUND(SUM(CASE WHEN f.status='Success' THEN f.amount_fcfa ELSE 0 END),0) AS valeur_fcfa,
    ROUND(AVG(CASE WHEN f.status='Success' THEN f.amount_fcfa END),0) AS montant_moyen_fcfa
FROM fact_transactions f
JOIN dim_client c
    ON f.client_id = c.client_id
GROUP BY c.client_segment
ORDER BY valeur_fcfa DESC;
"""
segment_sql = pd.read_sql_query(query, con)
segment_sql

## 1.7 Raisons d'échec

In [ ]:
query = """
SELECT
    failure_reason,
    COUNT(*) AS echecs
FROM fact_transactions
WHERE status='Failed'
GROUP BY failure_reason
ORDER BY echecs DESC;
"""
fail_sql = pd.read_sql_query(query, con)
fail_sql

## 1.8 Croissance mensuelle — CTE + LAG

In [ ]:
query = """
WITH mensuel AS (
    SELECT
        year_month,
        SUM(CASE WHEN status='Success' THEN amount_fcfa ELSE 0 END) AS valeur
    FROM fact_transactions
    GROUP BY year_month
),
evolution AS (
    SELECT
        year_month,
        valeur,
        LAG(valeur) OVER (ORDER BY year_month) AS valeur_prec
    FROM mensuel
)
SELECT
    year_month,
    ROUND(valeur,0) AS valeur_fcfa,
    ROUND(100.0 * (valeur - valeur_prec) / valeur_prec,2) AS croissance_mom_pct
FROM evolution
ORDER BY year_month;
"""
growth_sql = pd.read_sql_query(query, con)
growth_sql.tail(12)

# PARTIE 2 — Python / Pandas

Nous chargeons maintenant la vue analytique SQL dans Pandas pour approfondir l'analyse.


In [ ]:
df = pd.read_sql_query(
    "SELECT * FROM vw_mobile_money_analysis",
    con,
    parse_dates=["datetime_tx", "date_tx"]
)
df.head()


## 2.1 Contrôle qualité

In [ ]:
print("Dimensions :", df.shape)
print("Doublons transaction_id :", df["transaction_id"].duplicated().sum())
print("Valeurs manquantes hors failure_reason :", int(df.drop(columns=["failure_reason"]).isna().sum().sum()))
print("Période :", df["date_tx"].min().date(), "→", df["date_tx"].max().date())


## 2.2 KPI Python

In [ ]:
success = df["status"].eq("Success")
transactions = len(df)
success_tx = int(success.sum())
value_total = df.loc[success, "amount_fcfa"].sum()
fees = df["fee_fcfa"].sum()
commissions = df["agent_commission_fcfa"].sum()
net_revenue = df["net_revenue_fcfa"].sum()

kpi = pd.DataFrame({
    "KPI": ["Transactions","Transactions réussies","Taux de succès","Valeur traitée","Frais","Commissions agents","Revenu net"],
    "Valeur": [
        f"{transactions:,}",
        f"{success_tx:,}",
        f"{success_tx/transactions:.2%}",
        f"{value_total:,.0f} FCFA",
        f"{fees:,.0f} FCFA",
        f"{commissions:,.0f} FCFA",
        f"{net_revenue:,.0f} FCFA"
    ]
})
kpi


## 2.3 Performance opérateurs

In [ ]:
op_python = (
    df.assign(valeur_reussie=df["amount_fcfa"].where(success, 0))
      .groupby("operator", as_index=False)
      .agg(
          transactions=("transaction_id","count"),
          valeur_fcfa=("valeur_reussie","sum"),
          frais_fcfa=("fee_fcfa","sum"),
          revenu_net_fcfa=("net_revenue_fcfa","sum")
      )
)
op_python


## 2.4 Analyse des heures de pointe

In [ ]:
hourly = (
    df.groupby("hour", as_index=False)
      .agg(transactions=("transaction_id","count"),
           valeur_fcfa=("amount_fcfa","sum"))
      .sort_values("hour")
)
hourly.head()


## 2.5 Taux d'échec par canal

In [ ]:
channel_quality = (
    df.assign(echec=df["status"].eq("Failed").astype(int))
      .groupby("channel", as_index=False)
      .agg(transactions=("transaction_id","count"),
           echecs=("echec","sum"))
)
channel_quality["taux_echec"] = channel_quality["echecs"] / channel_quality["transactions"]
channel_quality.sort_values("taux_echec", ascending=False)


# PARTIE 3 — Visualisations

## 3.1 Valeur traitée par opérateur

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(op_python["operator"], op_python["valeur_fcfa"]/1_000_000)
plt.title("Valeur des transactions réussies par opérateur")
plt.xlabel("Opérateur")
plt.ylabel("Millions FCFA")
plt.tight_layout()
plt.show()


## 3.2 Activité horaire

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(hourly["hour"], hourly["transactions"], marker="o")
plt.title("Nombre de transactions par heure")
plt.xlabel("Heure")
plt.ylabel("Transactions")
plt.xticks(range(24))
plt.tight_layout()
plt.show()


## 3.3 Échecs par raison

In [ ]:
fail_plot = (
    df[df["status"].eq("Failed")]
      .groupby("failure_reason", as_index=False)
      .agg(echecs=("transaction_id","count"))
      .sort_values("echecs", ascending=True)
)
plt.figure(figsize=(9,5))
plt.barh(fail_plot["failure_reason"], fail_plot["echecs"])
plt.title("Principales raisons d'échec")
plt.xlabel("Nombre d'échecs")
plt.tight_layout()
plt.show()


# PARTIE 4 — Recommandations métier

Les analyses doivent conduire à des décisions :

1. **Qualité de service** — réduire les `Timeout réseau` et `Service indisponible`, particulièrement sur les canaux où le taux d'échec est supérieur.
2. **Réseau d'agents** — suivre séparément les volumes, commissions et niveaux d'activité des points agents.
3. **Revenu net** — ne pas analyser uniquement la valeur des transactions : certaines opérations peuvent avoir une forte valeur mais une faible contribution nette.
4. **Canaux digitaux** — suivre la migration USSD → Mobile App et comparer les taux de succès.
5. **Segments** — développer des offres adaptées aux PME et commerçants, qui peuvent afficher des montants moyens plus élevés.
6. **Géographie** — dimensionner le réseau d'agents et l'infrastructure selon les villes et les heures de pointe.
7. **Opérateurs** — les comparaisons MTN MoMo / Orange Money sont propres à ce jeu synthétique et ne constituent pas des parts de marché réelles.


# PARTIE 5 — Power BI

Le rapport Power BI comprend quatre pages :

1. **Executive Overview**
2. **Transactions & Revenue**
3. **Customers & Agents**
4. **Quality & Operations**

Les mesures DAX et le projet `.pbip` sont disponibles dans le dossier `powerbi/`.


In [ ]:
con.close()
print("Analyse terminée.")
